In [1]:

!pip install -q transformers librosa torchmetrics wandb

In [2]:

import os, random, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import librosa
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import (
    ASTFeatureExtractor,
    ASTForAudioClassification,
    get_cosine_schedule_with_warmup
)
from torchmetrics.classification import MulticlassF1Score
import wandb
warnings.filterwarnings('ignore')
print('All imports done!')

All imports done!


In [3]:

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
print('Seed set!')

Seed set!


In [ ]:


SAMPLE_RATE  = 16000
DURATION     = 20                        
MAX_LENGTH   = SAMPLE_RATE * DURATION

BATCH_SIZE   = 6                         
GRAD_ACCUM   = 2                         
NUM_WORKERS  = 2

EPOCHS_P1    = 3                         
EPOCHS_P2    = 5                        
LR_P1        = 3e-4                     
LR_P2        = 2e-5                      
LR_HEAD_MULT = 10                        
WEIGHT_DECAY = 0.01

TRAIN_SIZE   = 5000                      
NOISE_PROB   = 0.7                       
NOISE_LEVEL  = (0.03, 0.20)             
TEMPO_PROB   = 0.4                       
TEMPO_RANGE  = (0.88, 1.12)             
PATIENCE_P1  = 2                         
PATIENCE_P2  = 3                         
N_TTA        = 5                         

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device     : {DEVICE}')
print(f'Duration   : {DURATION}s')
print(f'Train size : {TRAIN_SIZE}')
print(f'Epochs P1  : {EPOCHS_P1}  |  Epochs P2: {EPOCHS_P2}')
print(f'LR P1      : {LR_P1}  |  LR P2: {LR_P2}')
print(f'TTA crops  : {N_TTA}')


WANDB_PROJECT = '23f2003621-t12026'          
WANDB_RUN_NAME = f'ast_p1{EPOCHS_P1}_p2{EPOCHS_P2}_dur{DURATION}s'  


Device     : cuda
Duration   : 20s
Train size : 5000
Epochs P1  : 3  |  Epochs P2: 5
LR P1      : 0.0003  |  LR P2: 2e-05
TTA crops  : 5


In [5]:

wandb.init(
    project = WANDB_PROJECT,
    name    = WANDB_RUN_NAME,
    config  = {
        'model'       : MODEL_NAME if 'MODEL_NAME' in dir() else 'MIT/ast-finetuned-audioset-10-10-0.4593',
        'duration'    : DURATION,
        'sample_rate' : SAMPLE_RATE,
        'batch_size'  : BATCH_SIZE,
        'grad_accum'  : GRAD_ACCUM,
        'epochs_p1'   : EPOCHS_P1,
        'epochs_p2'   : EPOCHS_P2,
        'lr_p1'       : LR_P1,
        'lr_p2'       : LR_P2,
        'lr_head_mult': LR_HEAD_MULT,
        'weight_decay': WEIGHT_DECAY,
        'train_size'  : TRAIN_SIZE,
        'noise_prob'  : NOISE_PROB,
        'tempo_prob'  : TEMPO_PROB,
        'n_tta'       : N_TTA,
        'patience_p1' : PATIENCE_P1,
        'patience_p2' : PATIENCE_P2,
    }
)
print(f'WandB run initialized: {wandb.run.name}')
print(f'View at: {wandb.run.url}')


wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

  2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

  ········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


WandB run initialized: ast_p13_p25_dur20s
View at: https://wandb.ai/S1mple02-team/23f2003621-t12026/runs/57gl4o2r


In [8]:

BASE_PATH      = '/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup'
GENRES_PATH    = os.path.join(BASE_PATH, 'genres_stems')
ESC_PATH       = os.path.join(BASE_PATH, 'ESC-50-master', 'audio')
REQUIRED_STEMS = ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']

GENRES   = ['blues','classical','country','disco','hiphop',
            'jazz','metal','pop','reggae','rock']
label2id = {g: i for i, g in enumerate(GENRES)}
id2label = {i: g for g, i in label2id.items()}
NUM_LABELS = len(GENRES)

print('Paths set!')
print('label2id:', label2id)

Paths set!
label2id: {'blues': 0, 'classical': 1, 'country': 2, 'disco': 3, 'hiphop': 4, 'jazz': 5, 'metal': 6, 'pop': 7, 'reggae': 8, 'rock': 9}


In [9]:

def build_song_index():
    index = {}
    for genre in GENRES:
        genre_path = os.path.join(GENRES_PATH, genre)
        valid = []
        for folder in sorted(os.listdir(genre_path)):
            folder_path = os.path.join(genre_path, folder)
            if not os.path.isdir(folder_path):
                continue
            
            if all(os.path.exists(os.path.join(folder_path, s))
                   for s in REQUIRED_STEMS):
                valid.append(folder_path)
        index[genre] = valid
        print(f'  {genre}: {len(valid)} songs')
    return index

song_index = build_song_index()


noise_files = []
for root, _, files in os.walk(ESC_PATH):
    for f in files:
        if f.endswith('.wav'):
            noise_files.append(os.path.join(root, f))

print(f'\nNoise files: {len(noise_files)}')

  blues: 100 songs
  classical: 100 songs
  country: 100 songs
  disco: 100 songs
  hiphop: 100 songs
  jazz: 100 songs
  metal: 100 songs
  pop: 100 songs
  reggae: 100 songs
  rock: 100 songs

Noise files: 2000


In [10]:

train_index = {}
val_index   = {}

for genre in GENRES:
    songs = song_index[genre][:]
    random.shuffle(songs)
    split = int(0.85 * len(songs))          # 85% train, 15% val
    train_index[genre] = songs[:split]
    val_index[genre]   = songs[split:]
    print(f'  {genre} → Train: {len(train_index[genre])}, Val: {len(val_index[genre])}')

print('\nTrain/Val split done!')

  blues → Train: 85, Val: 15
  classical → Train: 85, Val: 15
  country → Train: 85, Val: 15
  disco → Train: 85, Val: 15
  hiphop → Train: 85, Val: 15
  jazz → Train: 85, Val: 15
  metal → Train: 85, Val: 15
  pop → Train: 85, Val: 15
  reggae → Train: 85, Val: 15
  rock → Train: 85, Val: 15

Train/Val split done!


In [11]:

def load_audio(path):
    audio, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    return audio.astype(np.float32)

def normalize(audio):
    return audio / (np.max(np.abs(audio)) + 1e-6)

def crop_random(audio):
    """Random crop — used during training (augmentation)"""
    if len(audio) >= MAX_LENGTH:
        start = random.randint(0, len(audio) - MAX_LENGTH)
        return audio[start : start + MAX_LENGTH]
    return np.pad(audio, (0, MAX_LENGTH - len(audio)))

def crop_center(audio):
    """Centre crop — used during validation (reproducible)"""
    if len(audio) >= MAX_LENGTH:
        start = (len(audio) - MAX_LENGTH) // 2
        return audio[start : start + MAX_LENGTH]
    return np.pad(audio, (0, MAX_LENGTH - len(audio)))

def random_gain(audio):
    """Random volume change"""
    return audio * random.uniform(0.7, 1.3)

def tempo_augment(audio):
    """Random speed change — genre stays same but timing changes"""
    if random.random() < TEMPO_PROB:
        rate  = random.uniform(*TEMPO_RANGE)
        audio = librosa.effects.time_stretch(audio, rate=rate)
    return audio

def add_noise(audio):
    """Add ESC-50 environmental noise — simulates real mashups"""
    if random.random() < NOISE_PROB and len(noise_files) > 0:
        noise = load_audio(random.choice(noise_files))
        noise = crop_random(noise)
        level = random.uniform(*NOISE_LEVEL)
        audio = audio + level * noise
    return audio

print('Audio helpers ready!')

Audio helpers ready!


In [12]:

MODEL_NAME = 'MIT/ast-finetuned-audioset-10-10-0.4593'

feature_extractor = ASTFeatureExtractor.from_pretrained(MODEL_NAME)
print('Feature extractor loaded!')

model = ASTForAudioClassification.from_pretrained(
    MODEL_NAME,
    num_labels             = NUM_LABELS,
    id2label               = id2label,
    label2id               = label2id,
    ignore_mismatched_sizes = True      
)
model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f'AST model loaded on {DEVICE}')
print(f'Total parameters: {total_params:,}')

preprocessor_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

Feature extractor loaded!


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

ASTForAudioClassification LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                     | Status   |                                                                                        
------------------------+----------+----------------------------------------------------------------------------------------
classifier.dense.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([527, 768]) vs model:torch.Size([10, 768])
classifier.dense.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([527]) vs model:torch.Size([10])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


AST model loaded on cuda
Total parameters: 86,196,490


In [13]:

class TrainDataset(Dataset):
    """
    Generates TRAIN_SIZE synthetic mashups on the fly.
    Each sample: random stems from different songs of same genre
    + tempo augment + random gain + ESC-50 noise
    """
    def __init__(self, song_index, feature_extractor, size=TRAIN_SIZE):
        self.song_index        = song_index
        self.feature_extractor = feature_extractor
        self.size              = size

    def __len__(self):
        return self.size

    def __getitem__(self, idx):
        while True:
            try:
                
                genre = random.choice(GENRES)
                songs = self.song_index[genre]
                if len(songs) == 0:
                    continue

                
                mixed = None
                for stem in REQUIRED_STEMS:
                    song_path = random.choice(songs)
                    audio     = load_audio(os.path.join(song_path, stem))
                    audio     = crop_random(audio)
                    audio     = tempo_augment(audio)
                    audio     = random_gain(audio)
                    mixed = audio if mixed is None else mixed + audio

                mixed = normalize(mixed)
                mixed = add_noise(mixed)      
                mixed = normalize(mixed)

                
                inputs = self.feature_extractor(
                    mixed,
                    sampling_rate = SAMPLE_RATE,
                    return_tensors = 'pt'
                )
                return {
                    'input_values': inputs['input_values'].squeeze(0),
                    'labels'      : torch.tensor(label2id[genre], dtype=torch.long)
                }
            except Exception:
                continue   

print('TrainDataset class defined!')

TrainDataset class defined!


In [14]:

class ValDataset(Dataset):
    """
    Loads actual songs with centre crop — reproducible evaluation.
    No augmentation here.
    """
    def __init__(self, song_index, feature_extractor):
        self.feature_extractor = feature_extractor
        self.samples = []
        for genre in GENRES:
            for song_path in song_index[genre]:
                self.samples.append((genre, song_path))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        genre, song_path = self.samples[idx]
        mixed = None
        for stem in REQUIRED_STEMS:
            audio = load_audio(os.path.join(song_path, stem))
            audio = crop_center(audio)    # always same crop
            mixed = audio if mixed is None else mixed + audio

        mixed  = normalize(mixed)
        inputs = self.feature_extractor(
            mixed,
            sampling_rate  = SAMPLE_RATE,
            return_tensors = 'pt'
        )
        return {
            'input_values': inputs['input_values'].squeeze(0),
            'labels'      : torch.tensor(label2id[genre], dtype=torch.long)
        }

print('ValDataset class defined!')

ValDataset class defined!


In [15]:

train_dataset = TrainDataset(train_index, feature_extractor, size=TRAIN_SIZE)
val_dataset   = ValDataset(val_index, feature_extractor)

train_loader = DataLoader(
    train_dataset,
    batch_size        = BATCH_SIZE,
    shuffle           = True,
    num_workers       = NUM_WORKERS,
    pin_memory        = True,
    persistent_workers= True
)
val_loader = DataLoader(
    val_dataset,
    batch_size        = BATCH_SIZE,
    shuffle           = False,
    num_workers       = NUM_WORKERS,
    pin_memory        = True,
    persistent_workers= True
)

print(f'Train samples : {len(train_dataset)}')
print(f'Val samples   : {len(val_dataset)}')
print(f'Train batches : {len(train_loader)}')
print(f'Val batches   : {len(val_loader)}')

Train samples : 5000
Val samples   : 150
Train batches : 834
Val batches   : 25


In [16]:

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
f1_metric = MulticlassF1Score(
    num_classes = NUM_LABELS,
    average     = 'macro'
).to(DEVICE)

print('Loss: CrossEntropyLoss with label_smoothing=0.1')
print('Metric: Macro F1 Score')

Loss: CrossEntropyLoss with label_smoothing=0.1
Metric: Macro F1 Score


In [17]:

def train_one_epoch(model, loader, optimizer, scheduler):
    model.train()
    f1_metric.reset()
    total_loss = 0
    optimizer.zero_grad()

    for step, batch in enumerate(tqdm(loader, desc='Training')):
        input_values = batch['input_values'].to(DEVICE)
        labels       = batch['labels'].to(DEVICE)

        outputs = model(input_values=input_values)
        loss    = criterion(outputs.logits, labels) / GRAD_ACCUM
        loss.backward()

        
        if (step + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        total_loss += loss.item() * GRAD_ACCUM
        preds = torch.argmax(outputs.logits, dim=1)
        f1_metric.update(preds, labels)

    return total_loss / len(loader), f1_metric.compute().item()


def validate(model, loader):
    model.eval()
    f1_metric.reset()
    with torch.no_grad():
        for batch in tqdm(loader, desc='Validation'):
            input_values = batch['input_values'].to(DEVICE)
            labels       = batch['labels'].to(DEVICE)
            outputs      = model(input_values=input_values)
            preds        = torch.argmax(outputs.logits, dim=1)
            f1_metric.update(preds, labels)
    return f1_metric.compute().item()


print('train_one_epoch and validate functions ready!')

train_one_epoch and validate functions ready!


In [18]:

for param in model.base_model.parameters():
    param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Phase 1: Only {trainable:,} params trainable (classifier head only)')

optimizer_p1 = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr           = LR_P1,
    weight_decay = WEIGHT_DECAY
)
total_steps_p1 = (len(train_loader) // GRAD_ACCUM) * EPOCHS_P1
scheduler_p1   = get_cosine_schedule_with_warmup(
    optimizer_p1,
    num_warmup_steps   = total_steps_p1 // 10,
    num_training_steps = total_steps_p1
)

best_p1_f1   = 0
patience_cnt = 0

print(f'\n{"="*50}')
print(f'PHASE 1 — Classifier Warmup ({EPOCHS_P1} epochs)')
print(f'{"="*50}\n')

for epoch in range(EPOCHS_P1):
    train_loss, train_f1 = train_one_epoch(
        model, train_loader, optimizer_p1, scheduler_p1
    )
    val_f1 = validate(model, val_loader)

    print(f'Epoch {epoch+1}/{EPOCHS_P1}')
    print(f'  Train Loss : {train_loss:.4f}')
    print(f'  Train F1   : {train_f1:.4f}')
    print(f'  Val F1     : {val_f1:.4f}')

    wandb.log({
        'phase1/train_loss': train_loss,
        'phase1/train_f1'  : train_f1,
        'phase1/val_f1'    : val_f1,
        'phase1/epoch'     : epoch + 1,
    })

    if val_f1 > best_p1_f1:
        best_p1_f1 = val_f1
        torch.save(model.state_dict(), '/kaggle/working/best_model_phase1.pth')
        patience_cnt = 0
        print(f'  ✓ Saved best Phase 1 model (val F1: {best_p1_f1:.4f})')
    else:
        patience_cnt += 1
        if patience_cnt >= PATIENCE_P1:
            print('  Early stopping Phase 1.')
            break

print(f'\nPhase 1 done! Best val F1: {best_p1_f1:.4f}')

Phase 1: Only 9,226 params trainable (classifier head only)

PHASE 1 — Classifier Warmup (3 epochs)



Validation: 100%|██████████| 25/25 [00:55<00:00,  2.23s/it]


Epoch 1/3
  Train Loss : 1.2213
  Train F1   : 0.7215
  Val F1     : 0.7648
  ✓ Saved best Phase 1 model (val F1: 0.7648)


Validation: 100%|██████████| 25/25 [00:19<00:00,  1.26it/s]


Epoch 2/3
  Train Loss : 0.8141
  Train F1   : 0.8961
  Val F1     : 0.8241
  ✓ Saved best Phase 1 model (val F1: 0.8241)


Validation: 100%|██████████| 25/25 [00:19<00:00,  1.29it/s]

Epoch 3/3
  Train Loss : 0.7787
  Train F1   : 0.9093
  Val F1     : 0.8177

Phase 1 done! Best val F1: 0.8241


In [19]:

model.load_state_dict(torch.load('/kaggle/working/best_model_phase1.pth'))


for param in model.parameters():
    param.requires_grad = True

total_params = sum(p.numel() for p in model.parameters())
print(f'All {total_params:,} params unfrozen!')


optimizer_p2 = torch.optim.AdamW([
    {'params': model.base_model.parameters(),
     'lr'    : LR_P2},                          # tiny LR for pretrained
    {'params': model.classifier.parameters(),
     'lr'    : LR_P2 * LR_HEAD_MULT},           # 10x for classifier
], weight_decay=WEIGHT_DECAY)

total_steps_p2 = (len(train_loader) // GRAD_ACCUM) * EPOCHS_P2
scheduler_p2   = get_cosine_schedule_with_warmup(
    optimizer_p2,
    num_warmup_steps   = total_steps_p2 // 10,
    num_training_steps = total_steps_p2
)

best_p2_f1   = 0
patience_cnt = 0

print(f'\n{"="*50}')
print(f'PHASE 2 — Full Fine-Tune ({EPOCHS_P2} epochs)')
print(f'Base LR: {LR_P2}  |  Head LR: {LR_P2*LR_HEAD_MULT}')
print(f'{"="*50}\n')

for epoch in range(EPOCHS_P2):
    train_loss, train_f1 = train_one_epoch(
        model, train_loader, optimizer_p2, scheduler_p2
    )
    val_f1 = validate(model, val_loader)

    print(f'Epoch {epoch+1}/{EPOCHS_P2}')
    print(f'  Train Loss : {train_loss:.4f}')
    print(f'  Train F1   : {train_f1:.4f}')
    print(f'  Val F1     : {val_f1:.4f}')

    wandb.log({
        'phase2/train_loss': train_loss,
        'phase2/train_f1'  : train_f1,
        'phase2/val_f1'    : val_f1,
        'phase2/epoch'     : epoch + 1,
    })

    if val_f1 > best_p2_f1:
        best_p2_f1 = val_f1
        torch.save(model.state_dict(), '/kaggle/working/best_model_phase2.pth')
        wandb.run.summary['best_val_f1_p2'] = best_p2_f1
        patience_cnt = 0
        print(f'  ✓ Saved best Phase 2 model (val F1: {best_p2_f1:.4f})')
    else:
        patience_cnt += 1
        if patience_cnt >= PATIENCE_P2:
            print('  Early stopping Phase 2.')
            break

print(f'\nPhase 2 done! Best val F1: {best_p2_f1:.4f}')

All 86,196,490 params unfrozen!

PHASE 2 — Full Fine-Tune (5 epochs)
Base LR: 2e-05  |  Head LR: 0.0002



Validation: 100%|██████████| 25/25 [00:19<00:00,  1.28it/s]


Epoch 1/5
  Train Loss : 0.7543
  Train F1   : 0.9119
  Val F1     : 0.8158
  ✓ Saved best Phase 2 model (val F1: 0.8158)


Validation: 100%|██████████| 25/25 [00:19<00:00,  1.31it/s]


Epoch 2/5
  Train Loss : 0.6346
  Train F1   : 0.9554
  Val F1     : 0.8388
  ✓ Saved best Phase 2 model (val F1: 0.8388)


...

Epoch 3/5
  Train Loss : 0.5686
  Train F1   : 0.9789
  Val F1     : 0.8253


Validation: 100%|██████████| 25/25 [00:19<00:00,  1.26it/s]


Epoch 4/5
  Train Loss : 0.5414
  Train F1   : 0.9852
  Val F1     : 0.8314


Validation: 100%|██████████| 25/25 [00:19<00:00,  1.30it/s]

Epoch 5/5
  Train Loss : 0.5318
  Train F1   : 0.9885
  Val F1     : 0.8324
  Early stopping Phase 2.

Phase 2 done! Best val F1: 0.8388


In [21]:


def predict_with_tta(model, audio, n_tta=N_TTA):
    all_probs = []
    for _ in range(n_tta):
        cropped = crop_random(audio)       # different crop each time
        cropped = normalize(cropped)
        inputs  = feature_extractor(
            cropped,
            sampling_rate  = SAMPLE_RATE,  
            return_tensors = 'pt'
        )
        input_values = inputs['input_values'].to(DEVICE)
        with torch.no_grad():
            outputs = model(input_values=input_values)
            probs   = torch.softmax(outputs.logits, dim=1)
            all_probs.append(probs.cpu().numpy())

    avg_probs = np.mean(all_probs, axis=0)  # average N_TTA predictions
    return np.argmax(avg_probs, axis=1)[0]


print(f'TTA function ready! Using {N_TTA} crops per prediction.')


TTA function ready! Using 5 crops per prediction.


In [23]:


model.load_state_dict(torch.load('/kaggle/working/best_model_phase2.pth'))
model.eval()
print(f'Best model loaded! Val F1 was: {best_p2_f1:.4f}')

test_df   = pd.read_csv(os.path.join(BASE_PATH, 'test.csv'))
print(f'Test samples: {len(test_df)}')

all_ids   = []
all_preds = []

for idx in tqdm(range(len(test_df)), desc=f'Inference (TTA x{N_TTA})'):
    row   = test_df.iloc[idx]
    path  = os.path.join(BASE_PATH, row['filename'])
    audio = load_audio(path)
    pred  = predict_with_tta(model, audio, n_tta=N_TTA)
    all_ids.append(row['id'])
    all_preds.append(id2label[pred])

submission = pd.DataFrame({'id': all_ids, 'genre': all_preds})
submission.to_csv('/kaggle/working/submission.csv', index=False)

print(f'\nSubmission saved!')
print(f'Total predictions : {len(submission)}')
print(f'\nGenre distribution:')
print(submission['genre'].value_counts())
print(f'\nFirst 5 predictions:')
print(submission.head())

Best model loaded! Val F1 was: 0.8388
Test samples: 3020


Inference (TTA x5): 100%|██████████| 3020/3020 [25:31<00:00,  1.97it/s]


Submission saved!
Total predictions : 3020

Genre distribution:
genre
blues        425
rock         344
reggae       316
disco        316
jazz         313
hiphop       311
metal        308
pop          269
country      211
classical    207
Name: count, dtype: int64

First 5 predictions:
   id      genre
0   1        pop
1   2  classical
2   3      disco
3   4      metal
4   5    country


In [24]:

print('='*50)
print('TRAINING COMPLETE!')
print('='*50)
print(f'  Phase 1 best val F1 : {best_p1_f1:.4f}')
print(f'  Phase 2 best val F1 : {best_p2_f1:.4f}')
print(f'  TTA crops used      : {N_TTA}')
print(f'  Expected Kaggle F1  : {best_p2_f1 + 0.02:.4f} (approx)')
print('='*50)
print()
print('To improve further, try changing in Cell 4:')
print('  DURATION    = 25     (more audio context)')
print('  TRAIN_SIZE  = 6000   (more diversity)')
print('  EPOCHS_P2   = 7      (more fine-tuning)')
print('  LR_P2       = 1e-5   (safer updates)')
print('  N_TTA       = 7      (better inference)')

# Log final summary to WandB
wandb.run.summary['phase1_best_val_f1'] = best_p1_f1
wandb.run.summary['phase2_best_val_f1'] = best_p2_f1
wandb.run.summary['n_tta']              = N_TTA
wandb.finish()
print('WandB run finished!')


TRAINING COMPLETE!
  Phase 1 best val F1 : 0.8241
  Phase 2 best val F1 : 0.8388
  TTA crops used      : 5
  Expected Kaggle F1  : 0.8588 (approx)

To improve further, try changing in Cell 4:
  DURATION    = 25     (more audio context)
  TRAIN_SIZE  = 6000   (more diversity)
  EPOCHS_P2   = 7      (more fine-tuning)
  LR_P2       = 1e-5   (safer updates)
  N_TTA       = 7      (better inference)


phase1/epoch,▁▅█
phase1/train_f1,▁██
phase1/train_loss,█▂▁
phase1/val_f1,▁█▇
phase2/epoch,▁▃▅▆█
phase2/train_f1,▁▅▇██
phase2/train_loss,█▄▂▁▁
phase2/val_f1,▁█▄▆▆
best_val_f1_p2,0.83882
n_tta,5
phase1/epoch,3


WandB run finished!


In [25]:
submission.to_csv('/kaggle/working/submission.csv', index=False)

In [26]:
from IPython.display import FileLink

FileLink('/kaggle/working/submission.csv')

/kaggle/working/submission.csv

In [27]:
import shutil

shutil.make_archive('submission', 'zip', '/kaggle/working')

'/kaggle/working/submission.zip'